In [1]:
import os
from dotenv import load_dotenv


load_dotenv()

True

In [2]:
from langchain_community.document_loaders import PyPDFLoader

pdf_path = "telecom_guide.pdf"

loader = PyPDFLoader(pdf_path)
pages = loader.load()

print(f"Loaded {len(pages)} pages from {pdf_path}")
print(pages[0].page_content[0:100])

C:\Users\mukta\AppData\Local\Temp\ipykernel_23072\2379511654.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Loaded 9 pages from telecom_guide.pdf
Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Ca


In [3]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, 
    chunk_overlap=100, 
    separators=["\n\n", "\n", " . ", " ", ""]
)

chunks = splitter.split_documents(pages)

print(f"end={len(chunks)}")

end=24


In [4]:
print(f"chunks: {chunks[0].page_content}")

chunks: Telecom Technical Reference Guide  - Internal Use Only
Telecom Technical
Reference Guide
Customer Care & Network Operations Edition
Version 3.2  |  Covers 2G / 3G / 4G LTE / 5G
Page 1


In [5]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
vector_store = Chroma.from_documents(chunks, embeddings)

print(f"vectors stored: {vector_store._collection.count()}")

d:\Data\Tutorial\Tutorial FastAPI\9\learning-notebook-api\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5532.32it/s]


vectors stored: 24


In [6]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 2})

query = "What is VoLTE?"
response = retriever.invoke(query)


for i, document in enumerate(response, 1):
    print(f"Document {i}: {document.page_content}", end="\n\n")

Document 1: Telecom Technical Reference Guide  - Internal Use Only
6. VoLTE, VoWiFi, and Advanced Voice Services
Voice over LTE (VoLTE) and Voice over Wi-Fi (VoWiFi) are IP-based voice technologies that replace the legacy
circuit-switched voice channel used in 2G and 3G networks.
VoLTE: With VoLTE, voice calls are transmitted as data packets over the LTE network using the IMS (IP
Multimedia Subsystem) core. Benefits include HD voice quality (wideband audio at 16 kHz versus the 3.4 kHz of
legacy calls), faster call setup times (under 2 seconds versus 6-8 seconds on 3G), and the ability to use data and
voice simultaneously without degradation. VoLTE requires a compatible device, a VoLTE-enabled SIM, and an
account that has VoLTE activated.
Enabling VoLTE: On most Android devices navigate to Settings > Mobile Network > VoLTE and toggle it on. On
iPhone go to Settings > Mobile Data > Mobile Data Options > Voice & Data and select LTE. If the option is absent

Document 2: loss.
Fallback Beha

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_groq import ChatGroq


system_prompt = """
You are a helpful telecom assistant.
Answer the question using ONLY the context provided below.
If the context does not contain enough information, say so clearly.

Context:
{context}
"""

prompt = ChatPromptTemplate.from_messages([
    ("system", system_prompt),
    ("human", "{input}"),
])

llm = ChatGroq(
    model="openai/gpt-oss-20b",
    temperature=0.0,
    reasoning_format="parsed",
)

chain = (
    {
        "context": retriever, "input": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [8]:
response = chain.invoke("How does international roaming work and what charges should I expect?")
print(response)

**How international roaming works**

1. **Connection to a partner network**  
   When you travel outside your home network’s coverage area, your device automatically connects to a partner network in the visited country. This is called *roaming*.

2. **Authentication & authorization**  
   - The visited network authenticates you through an inter‑operator signalling protocol (SS7 or Diameter).  
   - Your home network validates your subscription and authorises the services you use.

3. **Traffic tunnelling**  
   All data, voice, and SMS traffic is tunneled back to your home network so that it can be billed correctly.  
   *Tunnelling adds latency compared to local network usage.*

4. **Roaming zones**  
   The world is divided into three roaming zones based on agreements and cost structures:  
   - **Zone A** – EU, UK, Australia, New Zealand (lowest roaming rates)  
   - **Zone B** – USA, Canada, Japan, Singapore (moderate rates)  
   - **Zone C** – Rest of the world (highest per‑MB and

In [9]:
while True:
    question = input("Ask something: ").strip()

    if question.lower() == "q":
        break

    for chunk in chain.stream(question):
        print(chunk, end="", flush=True)

    print()

**VoLTE (Voice over LTE)** is an IP‑based voice technology that replaces the legacy circuit‑switched voice channel used in 2G and 3G networks. It transmits voice calls as data packets over the LTE network using the IMS (IP Multimedia Subsystem) core.  

Key points from the guide:

- **HD voice quality** – wideband audio at 16 kHz (vs. 3.4 kHz on legacy calls).  
- **Faster call setup** – under 2 seconds (vs. 6–8 seconds on 3G).  
- **Simultaneous data and voice** – no degradation.  

To use VoLTE you need a compatible device, a VoLTE‑enabled SIM, and an account with VoLTE activated. On most Android devices, enable it via **Settings > Mobile Network > VoLTE**; on iPhone, go to **Settings > Mobile Data > Mobile Data Options > Voice & Data** and select LTE.
**5G Overview (from the guide)**  

- **Spectrum**: Operates across sub‑6 GHz bands and millimetre‑wave (mmWave) spectrum.  
- **Performance goals**:  
  - Peak data rates up to **10 Gbps**.  
  - Latency **under 1 ms**.  
  - Capacity

In [13]:
debug_question = "Modern 5G networks"

documents = retriever.invoke(debug_question)

print(f"retrieved: {documents}")

for i, document in enumerate(documents, 1):
    print(f"document: {document.page_content} (id: {i}")

retrieved: [Document(id='a6f13018-8d74-4a45-869a-494bc6d12009', metadata={'creationdate': '2026-04-30T14:16:24+00:00', 'producer': 'PyPDF', 'page': 1, 'creator': 'PyPDF', 'source': 'telecom_guide.pdf', 'total_pages': 9, 'page_label': '2'}, page_content='frequency bands, pushing speeds beyond 300 Mbps.\n5G networks represent the current frontier. Deploying across sub-6 GHz and millimetre-wave (mmWave)\nspectrum, 5G targets peak speeds of 10 Gbps, latency under 1 ms, and the ability to connect up to one million\ndevices per square kilometre. This density makes 5G critical for IoT deployments, autonomous vehicles, and\nsmart city infrastructure.\nNetwork coverage depends on the frequency band used. Lower frequencies (700 MHz, 800 MHz) travel farther\nand penetrate buildings better but carry less data. Higher frequencies (2.6 GHz, mmWave) carry more data but\nhave shorter range and are blocked more easily by walls and obstacles. Modern networks use a combination of\nbands to balance covera